# Pretrained Wide ResNet50-2 Backbone Notebook

This notebook evaluates a frozen ImageNet-pretrained `Wide ResNet50-2` backbone on the shared `x64` benchmark split. By default it reuses saved benchmark artifacts and only recomputes embeddings when explicitly requested.


## Submission Context

- Dataset notebook: `data/dataset/x64/benchmark_50k_5pct/notebook.ipynb`
- Dataset config: `data/dataset/x64/benchmark_50k_5pct/data_config.toml`
- Experiment config: `experiments/anomaly_detection/backbone_embedding/wide_resnet50_2/x64/baseline/train_config.toml`
- Artifact root: `experiments/anomaly_detection/backbone_embedding/wide_resnet50_2/x64/baseline/artifacts/wide_resnet50_2_embedding_baseline`
- Default behavior: load cached scores, summaries, and plots when they already exist; only re-extract embeddings when explicitly requested.
- Legacy note: older holdout-oriented Wide ResNet50-2 artifacts are preserved separately, but they are not the default submission path.


### Imports

This cell loads the libraries, repo-local modules, and path helpers used by the notebook.


In [ ]:
from pathlib import Path
import json
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.decomposition import PCA
from torch.utils.data import DataLoader

cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
REPO_ROOT = None
for candidate in candidate_roots:
    if (candidate / 'src' / 'wafer_defect').exists() and (candidate / 'configs').exists():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    raise RuntimeError('Could not locate repo root containing src/wafer_defect and configs/')

SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from wafer_defect.config import load_toml
from wafer_defect.data.wm811k import WaferMapDataset
from wafer_defect.evaluation import summarize_threshold_metrics, sweep_threshold_metrics


### Run Controls

This cell defines the experiment config path and the main rerun flags. Leave both flags `False` to reuse cached evaluation outputs when they already exist.


In [ ]:
CONFIG_PATH = REPO_ROOT / 'experiments/anomaly_detection/backbone_embedding/wide_resnet50_2/x64/baseline/train_config.toml'

# ── Run controls ───────────────────────────────────────────────────────────────
# RETRAIN = False          →  load saved evaluation artifacts (default)
# RETRAIN = True           →  re-extract all embeddings from scratch
# RERUN_PROJECTION = False →  load cached PCA projection (default)
# RERUN_PROJECTION = True  →  regenerate PCA projection even if cached
# REGENERATE_UMAP = False  →  display the pre-saved UMAP image (default)
# REGENERATE_UMAP = True   →  recompute UMAP from saved embeddings (requires umap-learn)
RETRAIN = False
RERUN_PROJECTION = False
REGENERATE_UMAP = False
PCA_MAX_POINTS_PER_GROUP = 500

config = load_toml(CONFIG_PATH)
config

### Reproducibility And Helpers

This cell sets the random seed, resolves the execution device, and defines the figure-saving helper used by the notebook.


In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def resolve_device(device_name: str) -> torch.device:
    if device_name == 'auto':
        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    return torch.device(device_name)


def save_figure(fig: plt.Figure, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, bbox_inches='tight', dpi=150)
    print(f'Saved figure to {path}')
    return path


set_seed(int(config['run']['seed']))
device = resolve_device(config['training'].get('device', 'auto'))
device


### Metadata And Data Loaders

This cell loads the configured metadata CSV, shows the split composition, and builds the loaders used if embeddings need to be recomputed.


In [ ]:
image_size = int(config['data'].get('image_size', 64))
batch_size = int(config['data'].get('batch_size', 64))
num_workers = int(config['data'].get('num_workers', 0))
metadata_path = REPO_ROOT / config['data']['metadata_csv']
metadata = pd.read_csv(metadata_path)

display(metadata.head())
display(metadata['split'].value_counts().rename_axis('split').to_frame('count'))
display(metadata['is_anomaly'].value_counts().rename_axis('is_anomaly').to_frame('count'))

train_dataset = WaferMapDataset(metadata_path, split='train', image_size=image_size)
val_dataset = WaferMapDataset(metadata_path, split='val', image_size=image_size)
test_dataset = WaferMapDataset(metadata_path, split='test', image_size=image_size)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

print(f'train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}')


### Model Builder And Scoring Helpers

This cell defines how the pretrained frozen backbone is built, how cached checkpoints are loaded when available, and how embeddings are scored.


In [ ]:
output_dir = REPO_ROOT / config['run']['output_dir']
eval_dir = output_dir / 'evaluation'
plots_dir = eval_dir / 'plots'
output_dir.mkdir(parents=True, exist_ok=True)
eval_dir.mkdir(parents=True, exist_ok=True)
plots_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = output_dir / 'wide_resnet50_2_backbone_baseline.pth'


def build_model() -> torch.nn.Module:
    try:
        from wafer_defect.models.resnet import ResNetFeatureExtractor
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            'torchvision is required only when embeddings need to be extracted. Install torchvision or keep FORCE_REEXTRACT = False to reuse cached artifacts.'
        ) from exc
    model = ResNetFeatureExtractor(
        backbone_name='wide_resnet50_2',
        pretrained=bool(config['model'].get('pretrained', True)),
        input_size=int(config['model'].get('input_size', 224)),
        freeze_backbone=bool(config['model'].get('freeze_backbone', True)),
        normalize_imagenet=bool(config['model'].get('normalize_imagenet', True)),
    ).to(device)
    if checkpoint_path.exists():
        checkpoint = torch.load(checkpoint_path, map_location=device)
        if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
            print(f'Loaded cached backbone checkpoint from {checkpoint_path}')
    return model


def collect_embeddings(model: torch.nn.Module, dataloader: DataLoader) -> tuple[np.ndarray, np.ndarray]:
    embeddings = []
    labels = []
    model.eval()
    with torch.inference_mode():
        for inputs, batch_labels in dataloader:
            inputs = inputs.to(device)
            batch_embeddings = model(inputs).cpu().numpy().astype(np.float32)
            embeddings.append(batch_embeddings)
            labels.append(batch_labels.numpy())
    return np.concatenate(embeddings, axis=0), np.concatenate(labels, axis=0)


def l2_center_scores(embeddings: np.ndarray, center: np.ndarray) -> np.ndarray:
    return np.linalg.norm(embeddings - center[None, :], axis=1)


output_dir


### Evaluation Cache Or Extraction

This cell either loads saved scores and summaries from the artifact folder or recomputes embeddings and saves a fresh evaluation bundle when explicitly requested.


In [ ]:
summary_path = eval_dir / 'summary.json'
val_scores_path = eval_dir / 'val_scores.csv'
test_scores_path = eval_dir / 'test_scores.csv'
threshold_sweep_path = eval_dir / 'threshold_sweep.csv'
train_center_path = output_dir / 'train_center.npy'
train_embeddings_path = eval_dir / 'train_embeddings.npy'
train_labels_path = eval_dir / 'train_labels.npy'
val_embeddings_path = eval_dir / 'val_embeddings.npy'
val_labels_path = eval_dir / 'val_labels.npy'
test_embeddings_path = eval_dir / 'test_embeddings.npy'
test_labels_path = eval_dir / 'test_labels.npy'

required_eval_paths = [summary_path, val_scores_path, test_scores_path, threshold_sweep_path, train_center_path]
embedding_cache_paths = [train_embeddings_path, train_labels_path, val_embeddings_path, val_labels_path, test_embeddings_path, test_labels_path]

train_embeddings = train_labels = val_embeddings = val_labels = test_embeddings = test_labels = None

if not RETRAIN and all(path.exists() for path in required_eval_paths):
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    val_scores_df = pd.read_csv(val_scores_path)
    test_scores_df = pd.read_csv(test_scores_path)
    threshold_sweep_df = pd.read_csv(threshold_sweep_path)
    train_center = np.load(train_center_path)
    threshold = float(summary['threshold'])
    metrics = summary['metrics_at_validation_threshold']
    best_sweep = summary['best_threshold_sweep']
    if all(path.exists() for path in embedding_cache_paths):
        train_embeddings = np.load(train_embeddings_path)
        train_labels = np.load(train_labels_path)
        val_embeddings = np.load(val_embeddings_path)
        val_labels = np.load(val_labels_path)
        test_embeddings = np.load(test_embeddings_path)
        test_labels = np.load(test_labels_path)
    print(f'Loaded saved artifacts from {output_dir}. Set RETRAIN = True to re-extract embeddings.')
else:
    model = build_model()
    train_embeddings, train_labels = collect_embeddings(model, train_loader)
    val_embeddings, val_labels = collect_embeddings(model, val_loader)
    test_embeddings, test_labels = collect_embeddings(model, test_loader)

    train_center = train_embeddings.mean(axis=0).astype(np.float32)
    val_scores = l2_center_scores(val_embeddings, train_center)
    test_scores = l2_center_scores(test_embeddings, train_center)

    val_scores_df = pd.DataFrame({'score': val_scores, 'is_anomaly': val_labels.astype(int)})
    test_scores_df = pd.DataFrame({'score': test_scores, 'is_anomaly': test_labels.astype(int)})

    threshold_quantile = float(config['scoring'].get('threshold_quantile', 0.95))
    val_normal_scores = val_scores_df.loc[val_scores_df['is_anomaly'] == 0, 'score']
    threshold = float(val_normal_scores.quantile(threshold_quantile))
    metrics = summarize_threshold_metrics(test_labels.astype(int), test_scores, threshold)
    threshold_sweep_df, best_sweep = sweep_threshold_metrics(test_labels.astype(int), test_scores)

    np.save(train_center_path, train_center)
    np.save(train_embeddings_path, train_embeddings)
    np.save(train_labels_path, train_labels.astype(np.int64))
    np.save(val_embeddings_path, val_embeddings)
    np.save(val_labels_path, val_labels.astype(np.int64))
    np.save(test_embeddings_path, test_embeddings)
    np.save(test_labels_path, test_labels.astype(np.int64))
    val_scores_df.to_csv(val_scores_path, index=False)
    test_scores_df.to_csv(test_scores_path, index=False)
    threshold_sweep_df.to_csv(threshold_sweep_path, index=False)

    checkpoint_payload = {
        'backbone_name': 'wide_resnet50_2',
        'config': config,
        'model_state_dict': model.state_dict(),
    }
    torch.save(checkpoint_payload, checkpoint_path)

    summary = {
        'backbone': 'wide_resnet50_2',
        'pretrained': bool(config['model'].get('pretrained', True)),
        'freeze_backbone': bool(config['model'].get('freeze_backbone', True)),
        'input_size': int(config['model'].get('input_size', 224)),
        'embedding_dim': int(train_embeddings.shape[1]),
        'train_center_norm': float(np.linalg.norm(train_center)),
        'threshold_quantile': threshold_quantile,
        'threshold': threshold,
        'checkpoint_path': str(checkpoint_path.relative_to(REPO_ROOT)),
        'metrics_at_validation_threshold': metrics,
        'best_threshold_sweep': best_sweep,
    }
    summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print(f'Saved evaluation outputs to {output_dir}')

summary

### Metrics

This cell displays the validation-threshold metrics and the best point on the threshold sweep.


In [ ]:
metrics_df = pd.DataFrame([
    {'metric': 'precision', 'value': metrics['precision']},
    {'metric': 'recall', 'value': metrics['recall']},
    {'metric': 'f1', 'value': metrics['f1']},
    {'metric': 'auroc', 'value': metrics['auroc']},
    {'metric': 'auprc', 'value': metrics['auprc']},
    {'metric': 'threshold', 'value': threshold},
])
display(metrics_df)
display(pd.DataFrame(metrics['confusion_matrix'], index=['true_normal', 'true_anomaly'], columns=['pred_normal', 'pred_anomaly']))
print(f"Best sweep threshold: {best_sweep['threshold']:.6f} | precision={best_sweep['precision']:.4f}, recall={best_sweep['recall']:.4f}, f1={best_sweep['f1']:.4f}")


### Score Plots

This cell rebuilds the main score-distribution and threshold-sweep figures from the saved evaluation tables, then saves the figures into the artifact folder.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(val_scores_df['score'], bins=30, alpha=0.8, color='#4d908e')
axes[0].axvline(threshold, color='red', linestyle='--', label=f'threshold={threshold:.4f}')
axes[0].set_title('Validation Normal Score Distribution')
axes[0].legend()

axes[1].hist(test_scores_df[test_scores_df['is_anomaly'] == 0]['score'], bins=30, alpha=0.7, label='normal')
axes[1].hist(test_scores_df[test_scores_df['is_anomaly'] == 1]['score'], bins=30, alpha=0.7, label='anomaly')
axes[1].axvline(threshold, color='red', linestyle='--', label=f'threshold={threshold:.4f}')
axes[1].set_title('Test Score Distribution')
axes[1].legend()
plt.tight_layout()
save_figure(fig, plots_dir / 'score_histograms.png')
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['precision'], label='precision')
ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['recall'], label='recall')
ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['f1'], label='f1')
ax.axvline(threshold, color='red', linestyle='--', label=f'validation threshold = {threshold:.4f}')
ax.axvline(best_sweep['threshold'], color='green', linestyle=':', label=f"best sweep threshold = {best_sweep['threshold']:.4f}")
ax.set_title('Threshold Sweep on Test Split')
ax.set_xlabel('Anomaly-score threshold')
ax.set_ylabel('Metric value')
ax.legend()
plt.tight_layout()
save_figure(fig, plots_dir / 'threshold_sweep_metrics.png')
plt.show()


### Embedding Projection

This cell loads or creates a compact PCA projection of the saved embeddings. If cached embeddings are missing, it explains how to regenerate them.


In [ ]:
projection_csv_path = eval_dir / 'embedding_pca_points.csv'
projection_png_path = plots_dir / 'embedding_pca.png'

if not RERUN_PROJECTION and projection_csv_path.exists():
    projection_df = pd.read_csv(projection_csv_path)
    print(f'Loaded cached PCA points from {projection_csv_path}')
elif all(path.exists() for path in embedding_cache_paths):
    if train_embeddings is None:
        train_embeddings = np.load(train_embeddings_path)
        train_labels = np.load(train_labels_path)
        val_embeddings = np.load(val_embeddings_path)
        val_labels = np.load(val_labels_path)
        test_embeddings = np.load(test_embeddings_path)
        test_labels = np.load(test_labels_path)
    rng = np.random.default_rng(int(config['run']['seed']))
    val_normal_idx = np.where(val_labels == 0)[0]
    test_normal_idx = np.where(test_labels == 0)[0]
    test_anomaly_idx = np.where(test_labels == 1)[0]
    sampled_val_normal = rng.choice(val_normal_idx, size=min(PCA_MAX_POINTS_PER_GROUP, len(val_normal_idx)), replace=False)
    sampled_test_normal = rng.choice(test_normal_idx, size=min(PCA_MAX_POINTS_PER_GROUP, len(test_normal_idx)), replace=False)
    sampled_test_anomaly = rng.choice(test_anomaly_idx, size=min(PCA_MAX_POINTS_PER_GROUP, len(test_anomaly_idx)), replace=False)
    pca_embeddings = np.concatenate([val_embeddings[sampled_val_normal], test_embeddings[sampled_test_normal], test_embeddings[sampled_test_anomaly]], axis=0)
    pca_labels = (['val_normal'] * len(sampled_val_normal) + ['test_normal'] * len(sampled_test_normal) + ['test_anomaly'] * len(sampled_test_anomaly))
    pca = PCA(n_components=2, random_state=int(config['run']['seed']))
    pca_points = pca.fit_transform(pca_embeddings)
    projection_df = pd.DataFrame({'pc1': pca_points[:, 0], 'pc2': pca_points[:, 1], 'group': pca_labels})
    projection_df.to_csv(projection_csv_path, index=False)
else:
    projection_df = None
    print('Embedding cache not found — PCA projection skipped. Set RETRAIN = True to regenerate embeddings.')

if projection_df is not None:
    fig, ax = plt.subplots(figsize=(7, 6))
    for group, color in [('val_normal', '#277da1'), ('test_normal', '#90be6d'), ('test_anomaly', '#f94144')]:
        subset = projection_df[projection_df['group'] == group]
        ax.scatter(subset['pc1'], subset['pc2'], s=18, alpha=0.6, label=group, color=color)
    ax.set_title('PCA of Wide ResNet50-2 Embeddings')
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.legend()
    plt.tight_layout()
    save_figure(fig, projection_png_path)
    plt.show()
    display(projection_df.head())

### UMAP Projection

This cell displays or recomputes the reference-fit UMAP of WideResNet50-2 patch embeddings.

- `REGENERATE_UMAP = False` (default): loads and displays the pre-saved `embedding_umap.png`
- `REGENERATE_UMAP = True`: recomputes UMAP using saved embedding `.npy` files via `umap_reference.py` (requires `umap-learn`; set `RETRAIN = True` first if embeddings are missing)

In [ ]:
from IPython.display import Image as IPImage

_UMAP_DIR = output_dir.parent / "umaps" / "wideresnet50A_embedding_baseline" / "evaluation"
_umap_png = _UMAP_DIR / "plots" / "embedding_umap.png"

if REGENERATE_UMAP:
    try:
        import umap as _umap_module
        from wafer_defect.evaluation.umap_reference import export_reference_umap_bundle

        _required = {
            "train": eval_dir / "train_embeddings.npy",
            "val":   eval_dir / "val_embeddings.npy",
            "val_labels": eval_dir / "val_labels.npy",
            "test":  eval_dir / "test_embeddings.npy",
            "test_labels": eval_dir / "test_labels.npy",
        }
        _missing = [k for k, p in _required.items() if not p.exists()]
        if _missing:
            raise FileNotFoundError(
                f"Missing embedding files: {_missing}. Set RETRAIN = True to regenerate them."
            )

        _UMAP_DIR.mkdir(parents=True, exist_ok=True)
        export_reference_umap_bundle(
            output_dir=_UMAP_DIR,
            umap_module=_umap_module,
            train_normal_embeddings=np.load(_required["train"]),
            val_embeddings=np.load(_required["val"]),
            val_labels=np.load(_required["val_labels"]),
            test_embeddings=np.load(_required["test"]),
            test_labels=np.load(_required["test_labels"]),
            pca_components=50,
            n_neighbors=15,
            min_dist=0.1,
            knn_k=15,
            metric="euclidean",
            random_state=int(config["run"]["seed"]),
        )
        print(f"UMAP regenerated and saved to {_UMAP_DIR}")
    except ImportError:
        print("umap-learn not available. Install with: pip install umap-learn")
    except FileNotFoundError as e:
        print(f"Cannot regenerate UMAP: {e}")

if _umap_png.exists():
    display(IPImage(filename=str(_umap_png)))
    print(f"UMAP: {_umap_png}")
else:
    print(f"UMAP image not found at {_umap_png}")
    print("Set REGENERATE_UMAP = True to compute it (requires umap-learn and embedding .npy files).")

### Failure Analysis

This cell attaches the model scores to the test metadata, summarizes the main error patterns, and saves the detailed failure-analysis tables.


In [ ]:
analysis_df = test_dataset.metadata.reset_index(drop=True).copy()
analysis_df['score'] = test_scores_df.reset_index(drop=True)['score']
analysis_df['predicted_anomaly'] = (analysis_df['score'] > threshold).astype(int)

analysis_df['error_type'] = 'tn'
analysis_df.loc[(analysis_df['is_anomaly'] == 0) & (analysis_df['predicted_anomaly'] == 1), 'error_type'] = 'fp'
analysis_df.loc[(analysis_df['is_anomaly'] == 1) & (analysis_df['predicted_anomaly'] == 0), 'error_type'] = 'fn'
analysis_df.loc[(analysis_df['is_anomaly'] == 1) & (analysis_df['predicted_anomaly'] == 1), 'error_type'] = 'tp'

error_summary_df = (
    analysis_df.groupby('error_type')
    .agg(count=('error_type', 'size'), mean_score=('score', 'mean'))
    .reindex(['tp', 'fn', 'fp', 'tn'])
)

defect_recall_df = (
    analysis_df[analysis_df['is_anomaly'] == 1]
    .groupby('defect_type')
    .agg(count=('defect_type', 'size'), detected=('predicted_anomaly', 'sum'), mean_score=('score', 'mean'))
    .sort_values(['detected', 'count'], ascending=[False, False])
)
defect_recall_df['recall'] = defect_recall_df['detected'] / defect_recall_df['count']

fp_defect_df = (
    analysis_df[analysis_df['error_type'] == 'fp']
    .groupby('defect_type')
    .agg(count=('defect_type', 'size'), mean_score=('score', 'mean'))
    .sort_values(['count', 'mean_score'], ascending=[False, False])
)

analysis_df.to_csv(eval_dir / 'failure_analysis.csv', index=False)
error_summary_df.to_csv(eval_dir / 'failure_error_summary.csv')
defect_recall_df.to_csv(eval_dir / 'failure_defect_recall.csv')
fp_defect_df.to_csv(eval_dir / 'failure_false_positive_breakdown.csv')

display(error_summary_df)
display(defect_recall_df)
display(fp_defect_df)
analysis_df.head()


### Failure Examples

This cell visualizes a few false positives and false negatives directly from the saved test metadata and score table, then saves the figure.


In [ ]:
def plot_failure_examples(error_type: str, n_examples: int = 6, score_order: str = 'desc') -> pd.DataFrame:
    subset = analysis_df[analysis_df['error_type'] == error_type].copy()
    if subset.empty:
        print(f'No samples found for error_type={error_type!r}.')
        return subset
    ascending = score_order == 'asc'
    subset = subset.sort_values('score', ascending=ascending).head(n_examples)
    fig, axes = plt.subplots(1, len(subset), figsize=(2.4 * len(subset), 2.8))
    if len(subset) == 1:
        axes = [axes]
    for axis, (sample_idx, row) in zip(axes, subset.iterrows()):
        wafer, _ = test_dataset[sample_idx]
        axis.imshow(wafer.squeeze(0), cmap='viridis')
        axis.set_title(f"{row.get('defect_type', 'unknown')}\nscore={row['score']:.3f}")
        axis.axis('off')
    plt.tight_layout()
    save_figure(fig, plots_dir / f'failure_examples_{error_type}.png')
    plt.show()
    return subset[['defect_type', 'score', 'predicted_anomaly', 'error_type']]

display(plot_failure_examples('fp', n_examples=6, score_order='desc'))
display(plot_failure_examples('fn', n_examples=6, score_order='asc'))


---

## Holdout Evaluation: Expanded 70k Normal / 3.5k Defect Test Set

This section evaluates the saved checkpoint on the secondary holdout split.

The holdout keeps the same 40k training normals and 5k validation normals as the main
benchmark, but replaces the test set with a much larger pool:
**70,000 normal + 3,500 defect** wafers. The `train_center` computed above is reused directly.

Set `RUN_HOLDOUT_EVALUATION = True` in the next cell to run.

- If cached holdout scores already exist, they are loaded without re-extracting embeddings.
- Set `FORCE_HOLDOUT_RERUN = True` to re-extract from scratch even if cached results exist.
- Embedding extraction requires torchvision and may take several minutes on CPU.

In [ ]:
# ── Holdout evaluation config ──────────────────────────────────────────────────
RUN_HOLDOUT_EVALUATION = False   # Set True to run
FORCE_HOLDOUT_RERUN = False      # Set True to re-extract even if cached results exist

HOLDOUT_METADATA_PATH = REPO_ROOT / 'data/processed/x64/wm811k/metadata_50k_5pct_holdout70k_3p5k.csv'
HOLDOUT_THRESHOLD_QUANTILE = 0.95
HOLDOUT_OUTPUT_DIR = output_dir / 'holdout70k_3p5k'
HOLDOUT_PLOTS_DIR = HOLDOUT_OUTPUT_DIR / 'plots'

print(f'Holdout metadata: {HOLDOUT_METADATA_PATH}')
print(f'Exists:           {HOLDOUT_METADATA_PATH.exists()}')
print(f'Output dir:       {HOLDOUT_OUTPUT_DIR}')

In [ ]:
if not RUN_HOLDOUT_EVALUATION:
    print('Holdout evaluation skipped. Set RUN_HOLDOUT_EVALUATION = True to run.')
else:
    HOLDOUT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    HOLDOUT_PLOTS_DIR.mkdir(parents=True, exist_ok=True)

    _h_summary_path = HOLDOUT_OUTPUT_DIR / 'summary.json'
    _h_val_scores_path = HOLDOUT_OUTPUT_DIR / 'val_scores.csv'
    _h_test_scores_path = HOLDOUT_OUTPUT_DIR / 'test_scores.csv'
    _h_sweep_path = HOLDOUT_OUTPUT_DIR / 'threshold_sweep.csv'
    _h_defect_path = HOLDOUT_OUTPUT_DIR / 'defect_recall.csv'
    _h_required = [_h_summary_path, _h_val_scores_path, _h_test_scores_path, _h_sweep_path]

    if not FORCE_HOLDOUT_RERUN and all(p.exists() for p in _h_required):
        # ── Load cached holdout results ───────────────────────────────────────
        _h_summary = json.loads(_h_summary_path.read_text(encoding='utf-8'))
        _h_val_scores_df = pd.read_csv(_h_val_scores_path)
        _h_test_scores_df = pd.read_csv(_h_test_scores_path)
        _h_sweep_df = pd.read_csv(_h_sweep_path)
        _h_threshold = float(_h_summary['threshold'])
        _h_metrics = _h_summary['metrics_at_validation_threshold']
        _h_best_sweep = _h_summary['best_threshold_sweep']
        print(f'Loaded cached holdout results from {HOLDOUT_OUTPUT_DIR}.')
        print('Set FORCE_HOLDOUT_RERUN = True to re-extract embeddings.')
    else:
        # ── Extract holdout embeddings ────────────────────────────────────────
        print('Extracting holdout embeddings (requires torchvision, may take several minutes)...')
        _h_model = build_model()

        _h_val_ds = WaferMapDataset(HOLDOUT_METADATA_PATH, split='val', image_size=image_size)
        _h_test_ds = WaferMapDataset(HOLDOUT_METADATA_PATH, split='test', image_size=image_size)
        _h_val_loader = DataLoader(_h_val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
        _h_test_loader = DataLoader(_h_test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
        print(f'  Val:  {len(_h_val_ds):,} wafers')
        print(f'  Test: {len(_h_test_ds):,} wafers')

        _h_val_emb, _h_val_labels = collect_embeddings(_h_model, _h_val_loader)
        _h_test_emb, _h_test_labels = collect_embeddings(_h_model, _h_test_loader)

        # ── Score using existing train_center ─────────────────────────────────
        _h_val_scores = l2_center_scores(_h_val_emb, train_center)
        _h_test_scores = l2_center_scores(_h_test_emb, train_center)

        _h_val_scores_df = pd.DataFrame({'score': _h_val_scores, 'is_anomaly': _h_val_labels.astype(int)})
        _h_test_scores_df = pd.DataFrame({'score': _h_test_scores, 'is_anomaly': _h_test_labels.astype(int)})

        # ── Threshold from val normals ────────────────────────────────────────
        _h_threshold = float(np.quantile(
            _h_val_scores[_h_val_labels == 0], HOLDOUT_THRESHOLD_QUANTILE
        ))

        # ── Metrics ───────────────────────────────────────────────────────────
        _h_metrics = summarize_threshold_metrics(_h_test_labels.astype(int), _h_test_scores, _h_threshold)
        _h_sweep_df, _h_best_sweep = sweep_threshold_metrics(_h_test_labels.astype(int), _h_test_scores)

        # ── Per-defect recall ─────────────────────────────────────────────────
        _h_test_meta = _h_test_ds.metadata.reset_index(drop=True).copy()
        _h_test_meta['score'] = _h_test_scores
        _h_test_meta['predicted'] = (_h_test_scores >= _h_threshold).astype(int)
        _defect_col = next((c for c in ['defect_type', 'failureType'] if c in _h_test_meta.columns), None)
        if _defect_col:
            _h_defect_df = (
                _h_test_meta[_h_test_meta['is_anomaly'] == 1]
                .groupby(_defect_col)
                .apply(lambda g: pd.Series({
                    'count': len(g),
                    'detected': int(g['predicted'].sum()),
                    'recall': g['predicted'].mean(),
                }))
                .reset_index()
                .sort_values('recall')
            )
            _h_defect_df.to_csv(_h_defect_path, index=False)

        # ── Save ──────────────────────────────────────────────────────────────
        _h_val_scores_df.to_csv(_h_val_scores_path, index=False)
        _h_test_scores_df.to_csv(_h_test_scores_path, index=False)
        _h_sweep_df.to_csv(_h_sweep_path, index=False)
        _h_summary = {
            'threshold': _h_threshold,
            'threshold_quantile': HOLDOUT_THRESHOLD_QUANTILE,
            'metrics_at_validation_threshold': _h_metrics,
            'best_threshold_sweep': _h_best_sweep,
        }
        _h_summary_path.write_text(json.dumps(_h_summary, indent=2), encoding='utf-8')
        print(f'Saved holdout results to {HOLDOUT_OUTPUT_DIR}')

    # ── Display results ───────────────────────────────────────────────────────
    print(f'\n── Holdout Evaluation Results (70k normal / 3.5k defect) ──────')
    print(f'  Threshold:           {_h_threshold:.6f}')
    print(f'  Precision:           {_h_metrics["precision"]:.6f}')
    print(f'  Recall:              {_h_metrics["recall"]:.6f}')
    print(f'  F1:                  {_h_metrics["f1"]:.6f}')
    print(f'  AUROC:               {_h_metrics["auroc"]:.6f}')
    print(f'  AUPRC:               {_h_metrics["auprc"]:.6f}')
    print(f'  Predicted anomalies: {_h_metrics.get("predicted_anomalies", "n/a")}')
    print(f'  Best sweep F1:       {_h_best_sweep["f1"]:.6f}')

    if _h_defect_path.exists():
        print(f'\n── Per-Defect Recall ────────────────────────────────────────────')
        display(pd.read_csv(_h_defect_path))

    # ── Plots ─────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].hist(_h_test_scores_df[_h_test_scores_df['is_anomaly'] == 0]['score'], bins=50, alpha=0.7, label='normal')
    axes[0].hist(_h_test_scores_df[_h_test_scores_df['is_anomaly'] == 1]['score'], bins=50, alpha=0.7, label='anomaly')
    axes[0].axvline(_h_threshold, color='red', linestyle='--', label=f'threshold={_h_threshold:.4f}')
    axes[0].set_title('Holdout Score Distribution')
    axes[0].set_xlabel('L2 distance from train center')
    axes[0].legend()
    axes[1].plot(_h_sweep_df['threshold'], _h_sweep_df['precision'], label='precision')
    axes[1].plot(_h_sweep_df['threshold'], _h_sweep_df['recall'], label='recall')
    axes[1].plot(_h_sweep_df['threshold'], _h_sweep_df['f1'], label='f1')
    axes[1].axvline(_h_threshold, color='red', linestyle='--', label=f'val threshold={_h_threshold:.4f}')
    axes[1].set_title('Holdout Threshold Sweep')
    axes[1].set_xlabel('Threshold')
    axes[1].legend()
    plt.tight_layout()
    save_figure(fig, HOLDOUT_PLOTS_DIR / 'holdout_distribution_sweep.png')
    plt.show()